In [1]:
from pystac_client import Client
import geopandas as gpd
import json
import xarray as xr
import zarr

In [3]:
import os
from pathlib import Path
root_dir = Path(os.path.abspath(""))
harz_boundaries = gpd.read_file(root_dir / "resources/Nationalpark_Harz_boundaries.geojson")
bbox_reproj = harz_boundaries.geometry.values[0].bounds

In [4]:
catalog = Client.open("https://stac.core.eopf.eodc.eu")

In [3]:
results = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_reproj,
    datetime=["2025-04-30", "2025-05-01"],
)
items = results.item_collection()[0]

In [4]:
ds = xr.open_datatree(
    items.assets["product"].href + "/measurements",
    engine="zarr",
    chunks={},
    decode_timedelta=True,
    consolidated=False,
)
ds

<xarray.DataTree>
Group: /

In [5]:
# Search with cloud cover filter
items = list(
    catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox_reproj,
        datetime=["2025-01-30", "2025-05-01"],
        query={"eo:cloud_cover": {"lt": 20}},  # Cloud cover less than 20%
    ).items()
)
print(items)

[<Item id=S2C_MSIL2A_20250428T103051_N0511_R108_T32UPC_20250428T160914>, <Item id=S2A_MSIL2A_20250427T101701_N0511_R065_T32UNC_20250427T170513>, <Item id=S2A_MSIL2A_20250420T103041_N0511_R108_T32UPC_20250420T202912>, <Item id=S2A_MSIL2A_20250420T103041_N0511_R108_T32UNC_20250420T202912>, <Item id=S2B_MSIL2A_20250420T101559_N0511_R065_T32UPC_20250420T131303>, <Item id=S2B_MSIL2A_20250420T101559_N0511_R065_T32UNC_20250420T131303>, <Item id=S2A_MSIL2A_20250407T101701_N0511_R065_T32UNC_20250407T171615>]


In [6]:
item = items[0]  # extracting the first item

ds = xr.open_dataset(
    item.assets["product"].href,
    **item.assets["product"].extra_fields["xarray:open_datatree_kwargs"],
)  # The engine="eopf-zarr" is already embedded in the STAC metadata

ds.quality_l2a_quicklook_r60m_tci.plot.imshow(rgb="quality_l2a_quicklook_r60m_band")

/home/randy/anaconda3/envs/eopf/lib/python3.12/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'eopf-zarr' loading failed:
cannot import name 'AggMethods' from 'xcube_resampling.constants' (/home/randy/anaconda3/envs/eopf/lib/python3.12/site-packages/xcube_resampling/constants.py)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


ValueError: unrecognized engine 'eopf-zarr' must be one of your download engines: ['netcdf4', 'h5netcdf', 'scipy', 'kerchunk', 'rasterio', 'store', 'zarr']. To install additional dependencies, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html